In [5]:
import numpy as np
import pandas as pd
import glob, sys, os, json

def extract_best_paramset_OneBasin_AllIter(outfile_optmz_paramset, path_archive, iterend, objfunc='nse', method='max', external_iter0_path=''):
    # extract the best parameter set based on all existing iterations 

    if os.path.isfile(outfile_optmz_paramset):
        return

    # find best iter/trial number
    df_allmet = []
    for iter in range(iterend+1):
        if iter==0 and len(external_iter0_path)>0:
            file_metric = f'{external_iter0_path}/outputs/iter{iter}_all_metric.csv'
        else:
            file_metric = f'{path_archive}/outputs/iter{iter}_all_metric.csv'
        dfi = pd.read_csv(file_metric)
        dfi['iter'] = iter
        dfi['trial'] = np.arange(len(dfi))
        df_allmet.append(dfi)
    df_allmet = pd.concat(df_allmet)

    if method == 'max':
        index = np.nanargmax(df_allmet[objfunc].values)
    elif method == 'min':
        index = np.nanargmin(df_allmet[objfunc].values)
        
    best_iter = df_allmet.iloc[index]['iter'].astype(np.int32)
    best_trial = df_allmet.iloc[index]['trial'].astype(np.int32)

    # copy best parameter set
    if best_iter == 0 and len(external_iter0_path)>0:
        file_param_source = f'{external_iter0_path}/param_sets/paramset_iter{best_iter}_trial{best_trial}.pkl'
    else:
        file_param_source = f'{path_archive}/param_sets/paramset_iter{best_iter}_trial{best_trial}.pkl'
        
    os.system(f'cp {file_param_source} {outfile_optmz_paramset}')

    source_info_file = outfile_optmz_paramset + '.sourceinfo'
    with open(source_info_file, 'w') as f:
        f.write(f'iter: {best_iter}\n')
        f.write(f'iter: {best_trial}\n')
    

In [6]:
subsetPath = f'/glade/campaign/cgd/tss/people/guoqiang/HBV_CAMELS/data_Yalan/Sub531ID.txt'
with open(subsetPath, 'r') as fp:
     camels_ids = json.load(fp)


In [7]:
path_archive = '/glade/campaign/cgd/tss/people/guoqiang/HBV_CAMELS/LSEexp2'
path_ext_iter0 = '/glade/campaign/cgd/tss/people/guoqiang/HBV_CAMELS/LSEexp1'
outpath = '/glade/campaign/cgd/tss/people/guoqiang/HBV_CAMELS/LSEexp2/BestParam_forShare'
os.makedirs(outpath, exist_ok=True)

iterend = 8

for id in camels_ids:
    p1 = f'{path_archive}/{id}'
    p2 = f'{path_ext_iter0}/{id}'
    
    outfile_optmz_paramset = f'{outpath}/BestParam_{id}.pkl'
    
    extract_best_paramset_OneBasin_AllIter(outfile_optmz_paramset, p1, iterend, objfunc='nse', method='max', external_iter0_path=p2)